In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
# base_dir = Path("/home/CAS_deepak/Noah_data_1982-2024_daily_mea")
# files = sorted(str(p) for p in base_dir.glob("sst.day.mean.*.nc"))
# ds = xr.open_mfdataset(files, combine="by_coords", parallel=True, engine="netcdf4", chunks={"time": 365})

# single-day map (first day of dataset)
day0 = ds.isel(time=0).sel(lat=slice(0, 30), lon=slice(40, 100))
plt.figure()
day0["sst"].plot()
plt.title(f"SST map on {str(ds['time'].isel(time=0).values)[:10]} (0–30N, 40–100E)")
plt.tight_layout()
plt.show()

# annual mean map for a chosen year
year_sel = 2019
annual = (ds.sel(time=str(year_sel)).sel(lat=slice(0, 30), lon=slice(40, 100))["sst"].mean("time", skipna=True))
plt.figure()
annual.plot()
plt.title(f"Annual mean SST in {year_sel} (0–30N, 40–100E)")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# convert res fields in to numpy arrays with explicit dtypes
idx_start = np.array(res["index_start"], dtype=int)
idx_end   = np.array(res["index_end"],   dtype=int)
duration  = np.array(res["duration"], dtype=int)
i_max     = np.array(res["intensity_max"], dtype=float)
i_mean    = np.array(res["intensity_mean"], dtype=float)
i_cum     = np.array(res["intensity_cumulative"], dtype=float)

n_events = idx_start.size
if n_events == 0:
    raise SystemExit("No events to tabulate; re-run detection or change region.")

# map indices back to real dates using time_dt 
start_dates = [pd.to_datetime(time_dt[i]).date() for i in idx_start]
end_dates   = [pd.to_datetime(time_dt[i]).date() for i in idx_end]

event_df = pd.DataFrame({"start_date": start_dates,"end_date":   end_dates,"duration_days": duration,
    "intensity_max_degC": i_max,"intensity_mean_degC": i_mean,"cumulative_intensity_degC": i_cum,})

#  read the dataframe
event_df["start_year"] = pd.to_datetime(event_df["start_date"]).dt.year
event_df["end_year"]   = pd.to_datetime(event_df["end_date"]).dt.year

# save the csv file
out_dir = Path.cwd() / "mhw_outputs"
out_dir.mkdir(exist_ok=True)
csv_path = out_dir / "mhw_events_boxmean_lat0-30_lon40-100.csv"
event_df.to_csv(csv_path, index=False)
print(f"Saved events table → {csv_path}")

# yearly summaries
yearly = (event_df.groupby("start_year").agg(events=("duration_days", "size"),total_hw_days=("duration_days", "sum"),
         mean_max_intensity=("intensity_max_degC", "mean")).reset_index().rename(columns={"start_year": "year"}))
print("First 5 rows of yearly summary:")
print(yearly.head())

# plot the figure
plt.figure()
plt.plot(yearly["year"], yearly["events"], marker="o")
plt.title("Marine heatwave events per year (box-mean 0–30N, 40–100E)")
plt.xlabel("Year"); plt.ylabel("Events")
plt.tight_layout(); plt.show()

plt.figure()
plt.plot(yearly["year"], yearly["total_hw_days"], marker="o")
plt.title("Total marine heatwave days per year (box-mean 0–30N, 40–100E)")
plt.xlabel("Year"); plt.ylabel("Days")
plt.tight_layout(); plt.show()



In [ ]:
from pathlib import Path
import xarray as xr
import numpy as np
import pandas as pd
import datetime as dt
import marineHeatWaves as mhw
import matplotlib.pyplot as plt

base_dir = Path("/home/Desktop/Noah_data_1982-2024_daily_mean/sst.day.mean.*.nc")  # update the path 

# collect files and open lazily with Dask
files = sorted(str(p) for p in glob(base_dir))
print(f"Found {len(files)} files")
if not files:
    raise FileNotFoundError(f"No files matched {base_dir}")

ds = xr.open_mfdataset(files,combine="by_coords", parallel=True, engine="netcdf4", chunks={"time": 365}) # adjust the chunk size for better memory usage 

# check the data
print("coords:", list(ds.coords))
print("data_vars:", list(ds.data_vars))
assert "sst" in ds.data_vars, "Expected variable 'sst' not found."

# subset the regional box 
ds_box = ds.sel(lat=slice(0, 30), lon=slice(40, 100))

# keep the sst unit in degC 
sst = ds_box["sst"]
units = (sst.attrs.get("units", "") or "").lower()
if units in ("k", "kelvin"):
    sst = sst - 273.15
    sst.attrs["units"] = "degC"
    print("Converted SST from Kelvin to degC")

# make a single time series by spatially averaging the regions
ts = sst.mean(dim=("lat", "lon"), skipna=True)

# convert time coordinate to Python datetime then to ORDINAL DAYS
def to_datetimeindex(dataarray):
    # try native conversion first
    try:
        return dataarray.indexes["time"].to_datetimeindex()
    except Exception:
        # calendars like noleap to convert to a standard proleptic gregorian timeline
        da2 = dataarray.convert_calendar("proleptic_gregorian")
        return da2.indexes["time"].to_datetimeindex()

time_dt = to_datetimeindex(ts)
time_ord = np.array([d.date().toordinal() for d in time_dt])  # WHAT MHW expects (ints)

temp = np.asarray(ts.values, dtype=float)

print("Series length:", temp.size, "range:", str(time_dt[0].date()), "→", str(time_dt[-1].date()))
print("Temp units:", sst.attrs.get("units", "unknown"))

# run marineHeatWaves detection on the box-mean series
res, clim = mhw.detect(time_ord, temp)

print("\n MHW detection summary of box-mean")
print("Events found:", len(res.get("index_start", [])))
if len(res.get("index_start", [])) > 0:
    i0 = 0
    print("First event: duration (days) =", int(res["duration"][i0]),
          "| max intensity (°C) =", float(res["intensity_max"][i0]))

# quick SST plotting
plt.figure()
plt.plot(time_dt, temp)
plt.title("Box-mean SST (lat 0–30, lon 40–100)")
plt.xlabel("Date")
plt.ylabel(f"SST ({sst.attrs.get('units', 'degC')})")
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np, 
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr

# extract seasonal climatology & threshold returned by MHW
seas   = np.asarray(clim["seas"], dtype=float)      # seasonal climatology (degC), length ~366
thresh = np.asarray(clim["thresh"], dtype=float)    # 90th percentile threshold (degC)

# build day-of-year (1 to 366) for plotting
doy_full = np.arange(1, len(seas) + 1)

# a sample year to overlay
year_sel = 2019
mask_y   = pd.DatetimeIndex(time_dt).year == year_sel
doy_y    = np.array([d.timetuple().tm_yday for d in time_dt[mask_y]])
temp_y   = temp[mask_y]

# climatology, threshold and  selected year SST
plt.figure()
plt.plot(doy_full, seas, label="Climatology (seas)")
plt.plot(doy_full, thresh, label="90th percentile threshold")
plt.scatter(doy_y, temp_y, s=6, label=f"SST in {year_sel} (box-mean)")
plt.title("Seasonal climatology & threshold vs SST")
plt.xlabel("Day of year"); plt.ylabel("°C"); plt.legend(); plt.tight_layout(); plt.show()

# show the first detected event on the full series
starts = np.asarray(res["index_start"], dtype=int)
ends   = np.asarray(res["index_end"],   dtype=int)
if starts.size:
    s0, e0 = starts[0], ends[0]
    plt.figure()
    plt.plot(time_dt, temp, lw=0.8)
    plt.axvspan(time_dt[s0], time_dt[e0], color="orange", alpha=0.3, label="First event")
    plt.title("Box-mean SST with first detected event highlighted")
    plt.xlabel("Date"); plt.ylabel("°C"); plt.legend(); plt.tight_layout(); plt.show()
